In [ ]:
# Thiết lập thư mục gốc một lần để notebook chạy được cả trên máy cá nhân và Kaggle.
import os
import sys
from pathlib import Path

_configured_root = os.environ.get("DATA_DIR")
if _configured_root:
    DATA_DIR = Path(_configured_root).expanduser().resolve()
else:
    DATA_DIR = next(
        (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
         if (p / "README.md").is_file() and (p / "data").is_dir()),
        None,
    )
    if DATA_DIR is None:
        raise FileNotFoundError("Set DATA_DIR to the repository root before running this notebook.")
os.environ["DATA_DIR"] = str(DATA_DIR)
os.chdir(DATA_DIR)
if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))


In [ ]:
import os
import sys
import json
import torch
import numpy as np
import pandas as pd
import random

def set_seed(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
set_seed(42)
print("Đã cố định random seed!")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {DEVICE}")

# 1. Nạp evaluator chuẩn
sys.path.append(str(DATA_DIR / "src/eval")) 
from evaluator import evaluate

# 2. Nạp bảng chỉ số chung và assert kiểm tra
with open(str(DATA_DIR / "data/embeddings/01_lightgcn_backbone_cold/item_index.json"), 'r') as f:
    course_ids = json.load(f)
with open(str(DATA_DIR / "data/embeddings/01_lightgcn_backbone_cold/user_index_cold.json"), 'r') as f:
    users = json.load(f)

assert len(course_ids) == 3769 and course_ids == sorted(course_ids), "Lỗi item_index.json"

# 3. Nạp embedding từ LightGCN nền chung đã sửa (g_i đã có g_i[cold_idx] = 0)
g_u = np.load(str(DATA_DIR / "data/embeddings/01_lightgcn_backbone_cold/g_u.npy"))
g_i = np.load(str(DATA_DIR / "data/embeddings/01_lightgcn_backbone_cold/g_i.npy"))

assert g_u.shape == (len(users), 64) and g_i.shape == (3769, 64), "Lỗi kích thước embedding nền chung"

# Đọc danh sách khóa lạnh val/test để xác định cold_idx
with open(str(DATA_DIR / "data/splits/cold_scenario/cold_items_val.json"), 'r') as f:
    cold_val_ids = json.load(f)
with open(str(DATA_DIR / "data/splits/cold_scenario/cold_items_test.json"), 'r') as f:
    cold_test_ids = json.load(f)

cold_val_ids = cold_val_ids.get("cold_items", cold_val_ids) if isinstance(cold_val_ids, dict) else cold_val_ids
cold_test_ids = cold_test_ids.get("cold_items", cold_test_ids) if isinstance(cold_test_ids, dict) else cold_test_ids

item2idx = {str(c): i for i, c in enumerate(course_ids)}
user2idx = {str(u): i for i, u in enumerate(users)}

cold_item_set = set(str(c) for c in cold_val_ids + cold_test_ids)
cold_idx = np.array(sorted(item2idx[c] for c in cold_item_set if c in item2idx))

# Kiểm tra assertion khóa lạnh bằng 0 - không có tín hiệu ID 
assert np.allclose(g_i[cold_idx], 0.0), "Lỗi: Khóa lạnh phải có embedding g_i = 0 đối với mô hình thuần ID!"

# 4. Chuẩn bị dữ liệu đánh giá tập test_cold
te_cold = pd.read_parquet(str(DATA_DIR / "data/splits/cold_scenario/test_cold.parquet"))
tr_cold = pd.read_parquet(str(DATA_DIR / "data/splits/cold_scenario/train_cold.parquet"))
va_cold = pd.read_parquet(str(DATA_DIR / "data/splits/cold_scenario/val_cold.parquet"))

user2id = user2idx
item2id = item2idx

target_cold = {}
for u, group in te_cold.groupby('user_id'):
    u_str = str(u)
    if u_str in user2id:
        u_idx = user2id[u_str]
        item_idxs = [item2id[str(i)] for i in group['course_id'] if str(i) in item2id]
        if len(item_idxs) > 0:
            target_cold[u_idx] = item_idxs

hist_cold_df = pd.concat([tr_cold, va_cold])
hist_cold = {}
for u, group in hist_cold_df.groupby('user_id'):
    u_str = str(u)
    if u_str in user2id:
        u_idx = user2id[u_str]
        item_idxs = [item2id[str(i)] for i in group['course_id'] if str(i) in item2id]
        if len(item_idxs) > 0:
            hist_cold[u_idx] = item_idxs

# Chuyển embedding sang Torch tensor trên Device
u_emb_t = torch.tensor(g_u, dtype=torch.float32, device=DEVICE)
i_emb_t = torch.tensor(g_i, dtype=torch.float32, device=DEVICE)

def score_fn_tid(users_tensor):
    u_idx = users_tensor.to(DEVICE)
    return torch.matmul(u_emb_t[u_idx], i_emb_t.t())

Đã cố định random seed!
Sử dụng thiết bị: cpu


In [3]:
# 5. Đánh giá Kịch bản (a): Chỉ ứng viên lạnh (kết quả kỳ vọng ~ Random với 5 seed phá hòa)
cand_cold_indices = [item2id[str(c)] for c in cold_test_ids if str(c) in item2id]
cand_bool_mask_a = np.zeros(3769, dtype=bool)
if len(cand_cold_indices) > 0:
    cand_bool_mask_a[cand_cold_indices] = True

print("\n--- Đánh giá LightGCN thuần ID trên Kịch bản Cold (a) với 5 seed ---")
SEEDS = [42, 2023, 2024, 1007, 999]
r20_a_list = []
results_a = []
for s in SEEDS:
    res_a = evaluate(score_fn_tid, target_cold, hist_cold, ks=(5, 10, 20), cand=cand_bool_mask_a, bs=1024, tie_seed=s)
    results_a.append(res_a)
    r20_a_list.append(res_a['Recall@20'].mean())

#for k in results_a[0].keys():
#    mean_value = np.mean([res[k].mean() for res in results_a])
#    print(f"{k}: {mean_value:.4f}")

for k in results_a[0].keys():
    seed_means = np.array([res[k].mean() for res in results_a])
    mean_value = seed_means.mean()
    std_value = seed_means.std(ddof=1)  
    print(f"{k}: {mean_value:.4f} ± {std_value:.4f}")
    
print(f"Recall@20 trung bình trên 5 seed (Kịch bản a): {np.mean(r20_a_list):.4f} (Kỳ vọng xấp xỉ mức ngẫu nhiên)")


--- Đánh giá LightGCN thuần ID trên Kịch bản Cold (a) với 5 seed ---
Recall@5: 0.0100 ± 0.0027
Recall@10: 0.0212 ± 0.0024
Recall@20: 0.0416 ± 0.0046
NDCG@5: 0.0067 ± 0.0019
NDCG@10: 0.0107 ± 0.0017
NDCG@20: 0.0165 ± 0.0022
HR@5: 0.0161 ± 0.0029
HR@10: 0.0344 ± 0.0023
HR@20: 0.0666 ± 0.0057
Recall@20 trung bình trên 5 seed (Kịch bản a): 0.0416 (Kỳ vọng xấp xỉ mức ngẫu nhiên)


In [4]:
# 6. Đánh giá Kịch bản (b): Toàn bộ không gian khóa học (Kết quả phải bằng 0 do khóa lạnh g_i = 0)
cand_bool_mask_b = np.ones(3769, dtype=bool)
res_cold_b = evaluate(score_fn_tid, target_cold, hist_cold, ks=(5, 10, 20), cand=cand_bool_mask_b, bs=1024)
print("\n--- Đánh giá LightGCN thuần ID trên Kịch bản Cold (b): Toàn bộ item ---")
for k, v in res_cold_b.items():
    print(f"{k}: {v.mean():.4f}")


--- Đánh giá LightGCN thuần ID trên Kịch bản Cold (b): Toàn bộ item ---
Recall@5: 0.0000
Recall@10: 0.0000
Recall@20: 0.0000
NDCG@5: 0.0000
NDCG@10: 0.0000
NDCG@20: 0.0000
HR@5: 0.0000
HR@10: 0.0000
HR@20: 0.0000
